In [1]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict

# Fix: use tf_keras to retain keras.preprocessing + utils APIs in this environment (keras==3 breaks these)
import tf_keras as keras
from tf_keras.layers import Dense, GlobalAveragePooling1D, Embedding
from tf_keras.callbacks import EarlyStopping
from tf_keras.models import Sequential
from tf_keras.preprocessing.sequence import pad_sequences
from tf_keras.preprocessing.text import Tokenizer
from tf_keras.utils import to_categorical

from sklearn.model_selection import train_test_split

np.random.seed(7)

INPUT_DIR = "/kaggle/input/spooky-author-identification"
TRAIN_PATH = os.path.join(INPUT_DIR, "train.csv")
TEST_PATH = os.path.join(INPUT_DIR, "test.csv")
SAMPLE_SUB_PATH = os.path.join(INPUT_DIR, "sample_submission.csv")

assert os.path.exists(TRAIN_PATH), f"Missing: {TRAIN_PATH}"
assert os.path.exists(TEST_PATH), f"Missing: {TEST_PATH}"
assert os.path.exists(SAMPLE_SUB_PATH), f"Missing: {SAMPLE_SUB_PATH}"



2026-01-23 19:41:42.295793: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769197302.308458      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769197302.312388      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-23 19:41:42.327886: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
df = pd.read_csv(TRAIN_PATH)

a2c = {"EAP": 0, "HPL": 1, "MWS": 2}
c2a = {v: k for k, v in a2c.items()}

y = np.array([a2c[a] for a in df.author], dtype=np.int64)
y = to_categorical(y, num_classes=3)

y.shape



(17621, 3)

In [3]:
counter = {name: defaultdict(int) for name in set(df.author)}
for text, author in zip(df.text, df.author):
    text = text.replace(" ", "")
    for c in text:
        counter[author][c] += 1

chars = set()
for v in counter.values():
    chars |= v.keys()

names = [author for author in counter.keys()]

print("c ", end="")
for n in names:
    print(n, end="   ")
print()
for c in chars:
    print(c, end=" ")
    for n in names:
        print(counter[n][c], end=" ")
    print()




c HPL   EAP   MWS   
K 156 75 30 
f 14538 19981 16537 
k 4746 3814 3338 
W 649 671 607 
u 17463 23540 18966 
J 193 145 60 
ñ 7 0 0 
â 0 6 0 
Z 43 15 2 
T 1423 1986 1121 
l 27162 31655 25074 
Æ 4 1 0 
Σ 1 0 0 
: 44 163 299 
g 13426 14459 11366 
i 39750 54488 41549 
n 45649 56062 45353 
p 9818 15618 11149 
R 209 233 348 
o 45784 60172 48209 
r 36412 45833 39732 
L 232 411 280 
F 236 337 219 
ü 5 1 0 
G 292 280 226 
I 3140 4339 4464 
v 5843 8571 7165 
, 7717 15651 10863 
" 461 2678 1337 
. 5332 7573 5206 
t 55884 73704 57079 
æ 10 32 0 
U 80 151 43 
? 153 457 368 
ç 0 1 0 
q 681 918 610 
N 316 368 182 
E 247 391 408 
S 757 651 521 
B 470 755 355 
h 38489 46016 39427 
C 390 372 275 
w 14050 15625 14485 
z 475 566 359 
è 0 13 0 
d 29982 32981 31906 
é 15 43 0 
Q 10 20 7 
ô 0 7 0 
H 657 791 606 
ë 10 0 0 
ö 2 12 0 
m 15879 20378 18513 
M 580 971 370 
P 287 382 314 
D 300 443 208 
Ν 1 0 0 
Å 1 0 0 
ï 7 0 0 
x 930 1766 1139 
Π 1 0 0 
X 5 14 3 
j 378 614 609 
e 79256 102699 88221 
s 39466 48003

In [4]:
def preprocess(text):
    text = text.replace("' ", " ' ")
    signs = set(',.:;"?!')
    prods = set(text) & signs
    if not prods:
        return text

    for sign in prods:
        text = text.replace(sign, " {} ".format(sign))
    return text




In [5]:
def create_docs(df, n_gram_max=2):
    def add_ngram(q, n_gram_max):
        ngrams = []
        for n in range(2, n_gram_max + 1):
            for w_index in range(len(q) - n + 1):
                ngrams.append("--".join(q[w_index : w_index + n]))
        return q + ngrams

    docs = []
    for doc in df.text:
        doc = preprocess(doc).split()
        docs.append(" ".join(add_ngram(doc, n_gram_max)))

    return docs




In [6]:
min_count = 2

docs = create_docs(df)
tokenizer = Tokenizer(lower=False, filters="")
tokenizer.fit_on_texts(docs)
num_words = sum([1 for _, v in tokenizer.word_counts.items() if v >= min_count])

tokenizer = Tokenizer(num_words=num_words, lower=False, filters="")
tokenizer.fit_on_texts(docs)
docs = tokenizer.texts_to_sequences(docs)
print(len(docs))



17621


In [7]:
maxlen = 256
docs = pad_sequences(sequences=docs, maxlen=maxlen)

docs.shape, tokenizer.num_words



((17621, 256), 70212)

In [8]:
# Keep core logic: input_dim computed from max token index + 1
input_dim = int(np.max(docs)) + 1
embedding_dims = 20
input_dim



70212

In [9]:
x_train, x_test, y_train, y_test = train_test_split(
    docs, y, test_size=0.20, random_state=42
)

x_train.shape, x_test.shape, y_train.shape, y_test.shape




((14096, 256), (3525, 256), (14096, 3), (3525, 3))

In [10]:
def create_model(embedding_dims=20, optimizer="adam"):
    model = Sequential()
    model.add(Embedding(input_dim=input_dim, output_dim=embedding_dims))
    model.add(GlobalAveragePooling1D())
    model.add(Dense(3, activation="softmax"))

    model.compile(
        loss="categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"]
    )
    return model




In [11]:
model = create_model()
model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, None, 20)          1404240   
                                                                 
 global_average_pooling1d (  (None, 20)                0         
 GlobalAveragePooling1D)                                         
                                                                 
 dense (Dense)               (None, 3)                 63        
                                                                 
Total params: 1404303 (5.36 MB)
Trainable params: 1404303 (5.36 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


2026-01-23 19:41:51.683735: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1769197311.684164      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:c6:00.0, compute capability: 8.6


In [12]:
# Keep training approach/loops; only ensure callback monitor name is valid for tf_keras.
epochs = 25

hist = model.fit(
    x_train,
    y_train,
    batch_size=16,
    validation_data=(x_test, y_test),
    epochs=epochs,
    callbacks=[
        EarlyStopping(patience=2, monitor="val_loss", restore_best_weights=True)
    ],
    verbose=2,
)



Epoch 1/25


I0000 00:00:1769197312.194357      64 service.cc:148] XLA service 0x7fa62c7d3260 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1769197312.194400      64 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-01-23 19:41:52.199386: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1769197312.212136      64 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1769197312.263124      64 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


881/881 - 25s - loss: 1.0750 - accuracy: 0.4083 - val_loss: 1.0574 - val_accuracy: 0.4553 - 25s/epoch - 28ms/step
Epoch 2/25
881/881 - 6s - loss: 0.9906 - accuracy: 0.5199 - val_loss: 0.9398 - val_accuracy: 0.5611 - 6s/epoch - 7ms/step
Epoch 3/25
881/881 - 3s - loss: 0.8207 - accuracy: 0.7197 - val_loss: 0.7898 - val_accuracy: 0.7308 - 3s/epoch - 4ms/step
Epoch 4/25
881/881 - 3s - loss: 0.6532 - accuracy: 0.8219 - val_loss: 0.6728 - val_accuracy: 0.7725 - 3s/epoch - 3ms/step
Epoch 5/25
881/881 - 2s - loss: 0.5274 - accuracy: 0.8585 - val_loss: 0.5950 - val_accuracy: 0.7994 - 2s/epoch - 3ms/step
Epoch 6/25
881/881 - 2s - loss: 0.4328 - accuracy: 0.8873 - val_loss: 0.5393 - val_accuracy: 0.8006 - 2s/epoch - 2ms/step
Epoch 7/25
881/881 - 2s - loss: 0.3593 - accuracy: 0.9103 - val_loss: 0.4973 - val_accuracy: 0.8278 - 2s/epoch - 2ms/step
Epoch 8/25
881/881 - 2s - loss: 0.2993 - accuracy: 0.9278 - val_loss: 0.4639 - val_accuracy: 0.8309 - 2s/epoch - 2ms/step
Epoch 9/25
881/881 - 2s - loss: 

In [13]:
test_df = pd.read_csv(TEST_PATH)
test_docs = create_docs(test_df)
test_docs = tokenizer.texts_to_sequences(test_docs)
test_docs = pad_sequences(sequences=test_docs, maxlen=maxlen)

# Fix: Keras 3 removed predict_proba; correct API is model.predict()
pred = model.predict(test_docs, batch_size=256, verbose=0)

pred.shape



(1958, 3)

In [14]:
# Ensure submission format/columns exactly: id,EAP,HPL,MWS
sub = pd.read_csv(SAMPLE_SUB_PATH)

# Align predictions to class columns in sample submission order
for col in ["EAP", "HPL", "MWS"]:
    sub[col] = pred[:, a2c[col]]

# Safety: ensure numeric and finite
sub[["EAP", "HPL", "MWS"]] = (
    sub[["EAP", "HPL", "MWS"]]
    .astype(float)
    .replace([np.inf, -np.inf], np.nan)
    .fillna(1 / 3)
)

out_path = "fastText_result_04.csv"
sub.to_csv(out_path, index=False)

print("Wrote:", out_path)
print(sub.head())

Wrote: fastText_result_04.csv
        id       EAP       HPL       MWS
0  id27251  0.000075  0.999922  0.000003
1  id09612  0.042797  0.024315  0.932888
2  id11943  0.001924  0.001347  0.996729
3  id19526  0.059595  0.101479  0.838926
4  id12931  0.046055  0.004862  0.949083
